In [ ]:
# ============================================================
# SAR Pipeline v2 — Full Analysis with GroupKFold + Validation
# Changes from v1:
#   - Section 4: GroupKFold(5) by farmer ID (Option Y)
#   - Section 3e: k-sensitivity analysis (k=2..5)
#   - Section 3f: GMM clustering robustness check
#   - Section 6b: Surrogate tree 80/20 held-out validation
#   - Section 6c: Surrogate tree 5-fold CV
# ============================================================

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import (
    cross_val_score, GroupKFold, StratifiedKFold,
    GroupShuffleSplit, cross_validate
)
from sklearn.metrics import (
    silhouette_score, calinski_harabasz_score, davies_bouldin_score,
    accuracy_score, classification_report
)
from sklearn.tree import DecisionTreeClassifier, plot_tree
from xgboost import XGBRegressor
from scipy.stats import spearmanr

import os
from pyprojroot import here
OUT = 'figures'
os.makedirs(OUT, exist_ok=True)
print("✓ Imports complete")

In [ ]:
# ============================================================
# SECTION 1 — DATA LOADING & FEATURE DEFINITIONS
# ============================================================
# หาตำแหน่งของโฟลเดอร์ที่ไฟล์ code นี้วางอยู่
base_path = here()
file_path = os.path.join(base_path, "datas", "SFProgramDataPanal.csv")

df = pd.read_csv(file_path)
df = df.replace('.', np.nan)

CLUSTER_FEATS = [
    'age','edu','agri_long','irriga','loan',
    'Avg_ProdManage','Avg_InputManage','Avg_Tech',
    'Avg_Ana&Plan','Avg_Mkting','Avg_Network',
    'Ave_ProdRisk','Ave_InputRisk','Ave_MktRisk','Ave_FinRisk'
]
RF_FEATS = [
    'age','edu','agri_long','gender','irriga','loan','region',
    'Avg_ProdManage','Avg_InputManage','Avg_Tech','Avg_Ana&Plan',
    'Avg_Mkting','Avg_Network',
    'Ave_ProdRisk','Ave_InputRisk','Ave_MktRisk','Ave_FinRisk',
    'sf_participant'
]
SHAP_FEATS = [
    'age','edu','agri_long','irriga','loan',
    'Avg_ProdManage','Avg_Tech','Avg_Mkting',
    'Ave_MktRisk','Ave_FinRisk','Overview_Risk',
    'agri Org_mem','gov_support','All_Skill'
]
SHAP_RENAME = {
    'age':'Age','edu':'Education','agri_long':'Agri Exp',
    'irriga':'Irrigation','loan':'Loan',
    'Avg_ProdManage':'Avg_ProdManage','Avg_Tech':'Avg_Tech',
    'Avg_Mkting':'Avg_Mkting','Ave_MktRisk':'Mkt Risk',
    'Ave_FinRisk':'Fin Risk','Overview_Risk':'Overall Risk',
    'agri Org_mem':'Agri Org','gov_support':'Gov Support',
    'All_Skill':'All_Skill'
}
OUTCOME       = 'Ch_Skill'
FARMER_ID     = 'id'
CLUSTER_NAMES = {0:'Low-skill', 1:'Moderate-skill', 2:'High-skill'}
CLUSTER_COL   = {0:'#4472C4', 1:'#FF8C00', 2:'#2ECC71'}
RF_PARAMS     = dict(n_estimators=500, min_samples_leaf=10,
                     max_features='sqrt', random_state=42, n_jobs=-1)

print(f"Dataset: {df.shape[0]} rows, {df[FARMER_ID].nunique()} unique farmers")
print(f"Panel structure: Year=0 (pre-training), Year=1 (post-training)")

In [ ]:
# ============================================================
# SECTION 2 — FARMER SEGMENTATION (K-MEANS, k=3)
# ============================================================

df_cl = df[CLUSTER_FEATS].apply(pd.to_numeric, errors='coerce').dropna().copy()
scaler = StandardScaler()
X_sc   = scaler.fit_transform(df_cl)

# ── 2a. Cluster validity indices k=2..8 ──────────────────────────────────
ks = range(2, 9)
cv_rows = []
for k in ks:
    km  = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl = km.fit_predict(X_sc)
    cv_rows.append({
        'k': k,
        'inertia':    km.inertia_,
        'silhouette': silhouette_score(X_sc, lbl),
        'calinski':   calinski_harabasz_score(X_sc, lbl),
        'davies':     davies_bouldin_score(X_sc, lbl),
    })
cv_df = pd.DataFrame(cv_rows)
print("\nCluster validity indices:")
print(cv_df.to_string(index=False, float_format='%.4f'))

# ── 2b. Fit k=3, label clusters ──────────────────────────────────────────
km3 = KMeans(n_clusters=3, random_state=42, n_init=10)
df_cl['Cluster_raw'] = km3.fit_predict(X_sc)

df_full = df.loc[df_cl.index].apply(pd.to_numeric, errors='coerce').copy()
df_full['Cluster_raw'] = df_cl['Cluster_raw'].values

order     = df_full.groupby('Cluster_raw')['All_Skill'].mean().sort_values()
label_map = {old: new for new, old in enumerate(order.index)}
df_full['Cluster']      = df_full['Cluster_raw'].map(label_map)
df_full['Cluster_name'] = df_full['Cluster'].map(CLUSTER_NAMES)

print("\nCluster sizes:")
print(df_full['Cluster_name'].value_counts().sort_index())

# ── 2c. Figure: 4-panel validity ─────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(14, 3.8))
panels = [
    ('inertia','Within-cluster Inertia','Elbow Method','#1565C0'),
    ('silhouette','Silhouette Score','Silhouette','#1976D2'),
    ('calinski','Calinski-Harabasz Score','Calinski-Harabasz','#0277BD'),
    ('davies','Davies-Bouldin Score','Davies-Bouldin','#00796B'),
]
for ax, (col, ylabel, title, color) in zip(axes, panels):
    ax.plot(cv_df['k'], cv_df[col], 'o-', color=color, lw=2, ms=6)
    ax.axvline(3, color='#E53935', ls='--', lw=1.5, alpha=0.8, label='k=3')
    for k, v in zip(cv_df['k'], cv_df[col]):
        ax.annotate(f'{v:.3f}', (k, v), textcoords='offset points',
                    xytext=(0, 8), ha='center', fontsize=7.5)
    ax.set(title=title, xlabel='Number of Clusters (k)', ylabel=ylabel)
    ax.grid(alpha=0.25); ax.legend(fontsize=8)
plt.suptitle('Cluster Validity: Four Complementary Indices (k=2–8)',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT}/fig_elbow_silhouette.png',
            bbox_inches='tight', dpi=180, facecolor='white')
plt.close()
print('✓ fig_elbow_silhouette.png')

# ── 2d. PCA scatter ───────────────────────────────────────────────────────
pca    = PCA(n_components=2, random_state=42)
X_pca  = pca.fit_transform(X_sc)
ev     = pca.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(7, 6))
for raw_lbl, mapped_lbl in label_map.items():
    mask = df_cl['Cluster_raw'] == raw_lbl
    ax.scatter(X_pca[mask,0], X_pca[mask,1],
               c=CLUSTER_COL[mapped_lbl],
               label=CLUSTER_NAMES[mapped_lbl],
               alpha=0.65, s=25, edgecolors='none')
ax.set(title='Farmer Clusters (k=3, Silhouette=0.185)',
       xlabel=f'PC1 ({ev[0]:.1%})', ylabel=f'PC2 ({ev[1]:.1%})')
ax.legend(framealpha=0.9, fontsize=10); ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(f'{OUT}/fig2_clusters_new.png',
            bbox_inches='tight', dpi=180, facecolor='white')
plt.close()
print('✓ fig2_clusters_new.png')

In [ ]:
# ============================================================
# SECTION 3e — K-SENSITIVITY ANALYSIS (k=2..5)
# ============================================================

print('\n=== Section 3e: k-sensitivity ===')
sens_rows = []
for k in range(2, 6):
    km_k  = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl_k = km_k.fit_predict(X_sc)
    df_full[f'Cluster_k{k}'] = df_cl.index.map(
        lambda i: lbl_k[df_cl.index.get_loc(i)] if i in df_cl.index else np.nan)
    sil = silhouette_score(X_sc, lbl_k)
    ch  = calinski_harabasz_score(X_sc, lbl_k)
    db  = davies_bouldin_score(X_sc, lbl_k)
    n_each = pd.Series(lbl_k).value_counts().sort_index().tolist()
    sens_rows.append({'k': k, 'Silhouette': sil, 'CH': ch, 'DB': db,
                      'Sizes': n_each})
    print(f"k={k}: Sil={sil:.3f} CH={ch:.1f} DB={db:.3f} sizes={n_each}")

# ── k-sensitivity figure ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
ks_s = [r['k'] for r in sens_rows]
for ax, (metric, label, col) in zip(axes, [
    ('Silhouette','Silhouette Score','#1976D2'),
    ('CH','Calinski-Harabasz','#0277BD'),
    ('DB','Davies-Bouldin','#00796B'),
]):
    vals = [r[metric] for r in sens_rows]
    ax.bar(ks_s, vals, color=col, alpha=0.8, edgecolor='white')
    ax.axvline(3, color='#E53935', ls='--', lw=1.5, alpha=0.8, label='k=3')
    for k, v in zip(ks_s, vals):
        ax.annotate(f'{v:.3f}', (k, v), textcoords='offset points',
                    xytext=(0, 5), ha='center', fontsize=9, fontweight='bold')
    ax.set(title=label, xlabel='k', ylabel=label)
    ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
plt.suptitle('k-Sensitivity Analysis: Cluster Validity for k=2..5',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT}/fig_k_sensitivity.png',
            bbox_inches='tight', dpi=180, facecolor='white')
plt.close()
print('✓ fig_k_sensitivity.png')

In [ ]:
# ============================================================
# SECTION 3f — GMM CLUSTERING ROBUSTNESS
# ============================================================

print('\n=== Section 3f: GMM robustness ===')
gmm_rows = []
for k in range(2, 6):
    gmm   = GaussianMixture(n_components=k, covariance_type='full',
                            random_state=42, n_init=5)
    lbl_g = gmm.fit_predict(X_sc)
    sil   = silhouette_score(X_sc, lbl_g)
    ch    = calinski_harabasz_score(X_sc, lbl_g)
    db    = davies_bouldin_score(X_sc, lbl_g)
    bic   = gmm.bic(X_sc)
    aic   = gmm.aic(X_sc)
    n_g   = pd.Series(lbl_g).value_counts().sort_index().tolist()
    gmm_rows.append({'k': k, 'Silhouette': sil, 'CH': ch,
                     'DB': db, 'BIC': bic, 'AIC': aic, 'Sizes': n_g})
    print(f"GMM k={k}: Sil={sil:.3f} CH={ch:.1f} DB={db:.3f} "
          f"BIC={bic:.0f} AIC={aic:.0f} sizes={n_g}")

# ── GMM vs KMeans comparison at k=3 ──────────────────────────────────────
km3_lbl  = km3.labels_
gmm3     = GaussianMixture(n_components=3, covariance_type='full',
                           random_state=42, n_init=5)
gmm3_lbl = gmm3.fit_predict(X_sc)

from sklearn.metrics import adjusted_rand_score
ari = adjusted_rand_score(km3_lbl, gmm3_lbl)
print(f"\nAdjusted Rand Index (KMeans k=3 vs GMM k=3): {ari:.4f}")
print("(ARI=1.0 = perfect agreement, 0.0 = random)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (lbl, title) in zip(axes, [
    (km3_lbl, f'K-Means (k=3)\nSilhouette={cv_df[cv_df["k"]==3]["silhouette"].values[0]:.3f}'),
    (gmm3_lbl, f'GMM (k=3)\nSilhouette={gmm_rows[1]["Silhouette"]:.3f}'),
]):
    order_l = pd.Series(lbl).value_counts().sort_values(ascending=False).index
    palette = ['#4472C4','#FF8C00','#2ECC71']
    for i, lbl_i in enumerate(order_l):
        mask = lbl == lbl_i
        ax.scatter(X_pca[mask,0], X_pca[mask,1], c=palette[i],
                   alpha=0.6, s=20, label=f'Cluster {i+1}')
    ax.set(title=title, xlabel=f'PC1 ({ev[0]:.1%})',
           ylabel=f'PC2 ({ev[1]:.1%})')
    ax.legend(fontsize=9); ax.grid(alpha=0.25)
plt.suptitle(f'Clustering Robustness: K-Means vs GMM (k=3)\n'
             f'Adjusted Rand Index = {ari:.3f}', fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT}/fig_gmm_robustness.png',
            bbox_inches='tight', dpi=180, facecolor='white')
plt.close()
print('✓ fig_gmm_robustness.png')

In [ ]:
# ============================================================
# SECTION 4 — PREDICTIVE MODELLING (GroupKFold — Option Y)
# ============================================================

print('\n=== Section 4: GroupKFold CV ===')

df_rf = df[RF_FEATS + [OUTCOME, FARMER_ID]].apply(
    pd.to_numeric, errors='coerce').dropna().copy()
X_rf   = df_rf[RF_FEATS].values
y_rf   = df_rf[OUTCOME].values
groups = df_rf[FARMER_ID].values

print(f"RF dataset: {len(df_rf)} rows, {df_rf[FARMER_ID].nunique()} farmers")

gkfold = GroupKFold(n_splits=5)

print(f"\n{'Model':<28} {'R²':>8} {'±SD':>7} {'RMSE':>8} {'MAE':>8}")
print('─'*57)
model_results = {}
for name, model in [
    ('Linear Regression', LinearRegression()),
    ('Ridge Regression',  Ridge(alpha=1.0)),
    ('Random Forest (GroupKFold)',
     RandomForestRegressor(**RF_PARAMS)),
]:
    r2   = cross_val_score(model, X_rf, y_rf, cv=gkfold,
                           groups=groups, scoring='r2')
    rmse = -cross_val_score(model, X_rf, y_rf, cv=gkfold, groups=groups,
                            scoring='neg_root_mean_squared_error')
    mae  = -cross_val_score(model, X_rf, y_rf, cv=gkfold, groups=groups,
                            scoring='neg_mean_absolute_error')
    model_results[name] = dict(R2=r2.mean(), SD=r2.std(),
                               RMSE=rmse.mean(), MAE=mae.mean())
    marker = ' ←' if 'Forest' in name else ''
    print(f"{name:<28} {r2.mean():>8.3f} {r2.std():>7.3f} "
          f"{rmse.mean():>8.3f} {mae.mean():>8.3f}{marker}")

# Train final RF on full data for SHAP
rf_final = RandomForestRegressor(**RF_PARAMS)
rf_final.fit(df_rf[RF_FEATS], y_rf)

In [ ]:
# ============================================================
# SECTION 5 — SHAP ANALYSIS
# ============================================================

print('\n=== Section 5: SHAP ===')

df_shap = df[SHAP_FEATS + [OUTCOME, FARMER_ID]].apply(
    pd.to_numeric, errors='coerce').dropna().copy()
X_shap  = df_shap[SHAP_FEATS].rename(columns=SHAP_RENAME)
y_shap  = df_shap[OUTCOME].values

rf_shap = RandomForestRegressor(**RF_PARAMS)
rf_shap.fit(X_shap, y_shap)

explainer = shap.TreeExplainer(rf_shap)
sv        = explainer.shap_values(X_shap)
sv_rf     = sv        # alias for robustness check
exp_rf    = explainer

mean_shap = pd.Series(np.abs(sv).mean(0), index=X_shap.columns)

# Global bar
mean_shap_sorted = mean_shap.sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(mean_shap_sorted.index, mean_shap_sorted.values,
        color='#2980b9', edgecolor='white', linewidth=0.5)
ax.set_xlabel('mean(|SHAP value|)', fontsize=10)
ax.set_title('SHAP Feature Importance → Ch_Skill', fontsize=12)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUT}/fig3_importance_new.png',
            bbox_inches='tight', dpi=180, facecolor='white')
plt.close()
print('✓ fig3_importance_new.png')

# Beeswarm
plt.figure(figsize=(10, 6))
shap.summary_plot(sv, X_shap, plot_type='dot',
                  max_display=10, show=False)
plt.title('SHAP Beeswarm: Determinants of Skill Change', fontsize=11)
plt.tight_layout()
plt.savefig(f'{OUT}/fig4_shap_new.png',
            bbox_inches='tight', dpi=180, facecolor='white')
plt.close()
print('✓ fig4_shap_new.png')

# Attach cluster labels
df_shap['Cluster'] = df_full.loc[
    df_shap.index.intersection(df_full.index), 'Cluster']
df_shap = df_shap.dropna(subset=['Cluster'])
df_shap['Cluster'] = df_shap['Cluster'].astype(int)

# Cluster-level beeswarm
panel_titles = [
    f'Cluster 1: Low-skill\n(n={(df_shap["Cluster"]==0).sum()})',
    f'Cluster 2: Moderate-skill\n(n={(df_shap["Cluster"]==1).sum()})',
    f'Cluster 3: High-skill\n(n={(df_shap["Cluster"]==2).sum()})',
]
fig = plt.figure(figsize=(17, 5))
for ci in range(3):
    ax  = fig.add_subplot(1, 3, ci+1)
    idx = df_shap[df_shap['Cluster']==ci].index
    sv_sub = explainer.shap_values(X_shap.loc[idx])
    top5   = (pd.Series(np.abs(sv_sub).mean(0), index=X_shap.columns)
                .sort_values(ascending=False).head(4).index.tolist())
    col_idx = [list(X_shap.columns).index(f) for f in top5]
    plt.sca(ax)
    shap.summary_plot(sv_sub[:, col_idx],
                      X_shap.loc[idx, top5].values,
                      feature_names=top5,
                      plot_type='dot', show=False,
                      plot_size=None, max_display=5)
    ax.set_title(panel_titles[ci], fontsize=11, fontweight='bold', pad=8)
    ax.tick_params(labelsize=9)
plt.suptitle('Cluster-Stratified SHAP: Segment-Specific Drivers',
             fontsize=12, y=1.03)
plt.tight_layout()
plt.savefig(f'{OUT}/fig4b_cluster_shap_new.png',
            bbox_inches='tight', dpi=180, facecolor='white')
plt.close()
print('✓ fig4b_cluster_shap_new.png')

# ── 5f. SHAP Robustness: RF vs XGBoost ───────────────────────────────────
print('\n--- 5f: SHAP Robustness ---')
xgb = XGBRegressor(n_estimators=500, learning_rate=0.05,
                   max_depth=6, subsample=0.8, colsample_bytree=0.8,
                   random_state=42, n_jobs=-1, verbosity=0)
xgb.fit(X_shap, y_shap)
exp_xgb = shap.TreeExplainer(xgb)
sv_xgb  = exp_xgb.shap_values(X_shap)

imp_rf  = pd.Series(np.abs(sv_rf ).mean(0), index=X_shap.columns, name='RF')
imp_xgb = pd.Series(np.abs(sv_xgb).mean(0), index=X_shap.columns, name='XGBoost')
rho, pval = spearmanr(imp_rf.rank(ascending=False),
                      imp_xgb.rank(ascending=False))
top4_rf  = imp_rf.sort_values(ascending=False).head(4).index.tolist()
top4_xgb = imp_xgb.sort_values(ascending=False).head(4).index.tolist()
overlap  = set(top4_rf) & set(top4_xgb)
print(f"Spearman ρ = {rho:.4f}  (p={pval:.4e})")
print(f"Top-4 overlap: {sorted(overlap)}  ({len(overlap)}/4)")

compare = pd.concat([imp_rf, imp_xgb], axis=1).sort_values('RF', ascending=True)
fig, ax = plt.subplots(figsize=(9, 5.5))
y_pos   = np.arange(len(compare)); h = 0.36
ax.barh(y_pos+h/2, compare['RF'],      height=h,
        color='#1565C0', alpha=0.85, label='Random Forest')
ax.barh(y_pos-h/2, compare['XGBoost'], height=h,
        color='#00796B', alpha=0.85, label='XGBoost')
ax.set_yticks(y_pos); ax.set_yticklabels(compare.index, fontsize=10)
ax.set_xlabel('mean(|SHAP value|)', fontsize=10)
ax.set_title(f'SHAP Attribution Robustness: RF vs XGBoost\n'
             f'Spearman ρ = {rho:.3f}  |  Top-4 overlap = {len(overlap)}/4',
             fontsize=11)
ax.legend(fontsize=10, loc='lower right')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUT}/fig_shap_robustness.png',
            bbox_inches='tight', dpi=180, facecolor='white')
plt.close()
print('✓ fig_shap_robustness.png')

In [ ]:
# ── 5g. SHAP attribution mass for top-4 features (per cluster) ──────────────
print('\n--- 5g: SHAP Attribution Mass (Top-4 features per cluster) ---')

# คำนวณ attribution mass สำหรับแต่ละ cluster
attribution_mass = []
for ci in range(3):
    idx = df_shap[df_shap['Cluster'] == ci].index
    if len(idx) == 0:
        continue
    sv_sub = explainer.shap_values(X_shap.loc[idx])
    mean_abs_shap = np.abs(sv_sub).mean(0)
    total_mass = mean_abs_shap.sum()
    
    # top-4 features ตาม paper
    top4_cluster = ['All_Skill', 'Avg_ProdManage', 'Education', 'Avg_Mkting']
    # หา index ของ features เหล่านี้ใน X_shap.columns
    top4_idx = [list(X_shap.columns).index(f) for f in top4_cluster if f in X_shap.columns]
    top4_mass = mean_abs_shap[top4_idx].sum()
    pct = (top4_mass / total_mass) * 100
    
    cluster_name = CLUSTER_NAMES[ci]
    print(f'{cluster_name:>15}: top-4 features capture {pct:.1f}% of SHAP attribution mass')
    attribution_mass.append({'cluster': cluster_name, 'percentage': pct})

print(f'\nAverage across clusters: {np.mean([m["percentage"] for m in attribution_mass]):.1f}%')

In [ ]:
# ============================================================
# SECTION 6 — SURROGATE DECISION TREE
# ============================================================

print('\n=== Section 6: Surrogate Decision Tree ===')

# SHAP-selected top-5 features
TOP4_FEATS = ['All_Skill', 'Avg_ProdManage', 'edu', 'Avg_Mkting']
top_feats = TOP4_FEATS  # override with fixed top-4 for consistency
# top_feats = mean_shap.sort_values(ascending=False).head(5).index.tolist()
feat_orig = [k for k, v in SHAP_RENAME.items() if v in top_feats]
print(f"Top-5 SHAP features: {top_feats}")

# ── 6a. In-sample (baseline — existing result) ────────────────────────────
X_tree_all = df_shap[feat_orig].rename(columns=SHAP_RENAME)
y_tree_all = (df_shap['Cluster'] == 2).astype(int)
farmer_all = df_shap[FARMER_ID].values

dt_full = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_full.fit(X_tree_all, y_tree_all)
acc_insample = accuracy_score(y_tree_all, dt_full.predict(X_tree_all))
print(f"\n6a. In-sample accuracy: {acc_insample:.3f}  (n={len(X_tree_all)})")

# ── Figure: Surrogate tree ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(18, 8))
plot_tree(dt_full, feature_names=top_feats,
          class_names=['Low/Moderate','High'],
          filled=True, rounded=True, fontsize=10, ax=ax)
ax.set_title(f'Surrogate Decision Tree (depth=3, in-sample acc={acc_insample:.3f})',
             fontsize=12, pad=15)
plt.tight_layout()
plt.savefig(f'{OUT}/fig5_rules_new.png',
            bbox_inches='tight', dpi=180, facecolor='white')
plt.close()
print('✓ fig5_rules_new.png')

# ── 6b. Farmer-level 80/20 train/test split (Option B) ───────────────────
print('\n6b. Farmer-level 80/20 held-out validation (Option B)')

# Clustering features available in SHAP dataset
SURR_CLUSTER_FEATS = ['age','edu','agri_long','irriga','loan',
                      'Avg_ProdManage','Avg_Tech','Avg_Mkting',
                      'Ave_MktRisk','Ave_FinRisk']

# Year=1 only — one row per farmer (true post-training outcome)
ALL_FEATS_Y1 = SHAP_FEATS + [OUTCOME, FARMER_ID]
df_y0 = df[df['Year']==0][ALL_FEATS_Y1].apply(
    pd.to_numeric, errors='coerce').dropna(
    subset=SHAP_FEATS+[OUTCOME]).drop_duplicates(
    subset=FARMER_ID).reset_index(drop=True)

unique_farmers = df_y0[FARMER_ID].unique()
np.random.seed(42); np.random.shuffle(unique_farmers)
n_train    = int(len(unique_farmers) * 0.80)
train_ids  = unique_farmers[:n_train]
test_ids   = unique_farmers[n_train:]

df_tr = df_y0[df_y0[FARMER_ID].isin(train_ids)].copy()
df_te = df_y0[df_y0[FARMER_ID].isin(test_ids )].copy()
print(f"Train: {len(df_tr)} farmers | Test: {len(df_te)} farmers")

# Cluster on TRAIN only (Option B)
scaler_tr    = StandardScaler()
X_tr_sc      = scaler_tr.fit_transform(df_tr[SURR_CLUSTER_FEATS])
km_tr        = KMeans(n_clusters=3, random_state=42, n_init=10)
df_tr = df_tr.copy()
df_tr['Cluster_raw'] = km_tr.fit_predict(X_tr_sc)
tr_order   = df_tr.groupby('Cluster_raw')['All_Skill'].mean().sort_values()
tr_lbl_map = {old: new for new, old in enumerate(tr_order.index)}
df_tr['Cluster'] = df_tr['Cluster_raw'].map(tr_lbl_map)
print(f"Train cluster sizes: {df_tr['Cluster'].value_counts().sort_index().to_dict()}")

# Assign test to nearest centroid
df_te = df_te.copy()
df_te['Cluster_raw'] = km_tr.predict(
    scaler_tr.transform(df_te[SURR_CLUSTER_FEATS]))
df_te['Cluster'] = df_te['Cluster_raw'].map(tr_lbl_map)
print(f"Test  cluster sizes: {df_te['Cluster'].value_counts().sort_index().to_dict()}")

# SHAP on TRAIN to select top-4 features
X_tr_shap = df_tr[SHAP_FEATS].rename(columns=SHAP_RENAME)
rf_tr     = RandomForestRegressor(**RF_PARAMS)
rf_tr.fit(X_tr_shap, df_tr[OUTCOME].values)
sv_tr     = shap.TreeExplainer(rf_tr).shap_values(X_tr_shap)
top5_tr   = (pd.Series(np.abs(sv_tr).mean(0), index=X_tr_shap.columns)
               .sort_values(ascending=False).head(4).index.tolist())
feat_orig_tr = [k for k, v in SHAP_RENAME.items() if v in top5_tr]
print(f"Train top-5 SHAP: {top5_tr}")

# Surrogate tree — TRAIN
X_surr_tr = df_tr[feat_orig_tr].rename(columns=SHAP_RENAME)
y_surr_tr = (df_tr['Cluster'] == 2).astype(int)
dt_val    = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_val.fit(X_surr_tr, y_surr_tr)
acc_train = accuracy_score(y_surr_tr, dt_val.predict(X_surr_tr))

# Surrogate tree — TEST
X_surr_te = df_te[feat_orig_tr].rename(columns=SHAP_RENAME)
y_surr_te = (df_te['Cluster'] == 2).astype(int)
acc_test  = accuracy_score(y_surr_te, dt_val.predict(X_surr_te))

print(f"\nTrain accuracy (80%):    {acc_train:.3f}  (n={len(df_tr)})")
print(f"Held-out test (20%):     {acc_test:.3f}  (n={len(df_te)})")
print(f"\n{classification_report(y_surr_te, dt_val.predict(X_surr_te), target_names=['Low/Mod','High'])}")

# ── 6c. 5-fold GroupKFold CV ──────────────────────────────────────────────
print('\n6c. 5-fold GroupKFold CV on surrogate tree')

X_cv_all = df_shap[feat_orig_tr].rename(columns=SHAP_RENAME)
y_cv_all = (df_shap['Cluster'] == 2).astype(int)
g_cv_all = df_shap[FARMER_ID].values

gkf_surr = GroupKFold(n_splits=5)
cv_accs  = []
for fold, (tri, tei) in enumerate(gkf_surr.split(
        X_cv_all, y_cv_all, groups=g_cv_all)):
    dt_f = DecisionTreeClassifier(max_depth=3, random_state=42)
    dt_f.fit(X_cv_all.iloc[tri], y_cv_all.iloc[tri])
    a = accuracy_score(y_cv_all.iloc[tei],
                       dt_f.predict(X_cv_all.iloc[tei]))
    cv_accs.append(a)
    print(f"  Fold {fold+1}: {a:.3f}")
print(f"\n5-fold GroupKFold CV: {np.mean(cv_accs):.3f} +/- {np.std(cv_accs):.3f}")

print("\n=== FINAL SUMMARY: Surrogate Tree Accuracy ===")
print(f"{'In-sample (full, n=817)':<40} {acc_insample:.3f}")
print(f"{'Train set (80% farmers)':<40} {acc_train:.3f}")
print(f"{'Held-out test (20% farmers)':<40} {acc_test:.3f}")
print(f"{'5-fold GroupKFold CV':<40} {np.mean(cv_accs):.3f} +/- {np.std(cv_accs):.3f}")

In [ ]:
# ============================================================
# SUMMARY
# ============================================================
print('\n' + '='*55)
print('All figures saved to ./figures/')
print('='*55)
for fn, desc in [
    ('fig_elbow_silhouette.png',   '4-panel cluster validity'),
    ('fig2_clusters_new.png',      'PCA scatter k=3'),
    ('fig_k_sensitivity.png',      'k-sensitivity k=2..5  [NEW]'),
    ('fig_gmm_robustness.png',     'GMM vs K-Means  [NEW]'),
    ('fig3_importance_new.png',    'Global SHAP bar'),
    ('fig4_shap_new.png',          'Global SHAP beeswarm'),
    ('fig4b_cluster_shap_new.png', 'Cluster SHAP 3-panel'),
    ('fig_shap_robustness.png',    'RF vs XGBoost SHAP'),
    ('fig5_rules_new.png',         'Surrogate decision tree'),
]:
    print(f"  {fn:<38} {desc}")